In [1]:
from datasets import load_dataset, Dataset as HFDataset
from itertools import islice
from PIL import Image
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn
import os
from torch.utils.data import IterableDataset
from PIL import Image
import torchvision.transforms as transforms

/opt/miniconda3/envs/image_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
weights= ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

model.fc = nn.Identity()
model = model.to(device)
model.eval()

preprocess = weights.transforms()

Using device: mps


In [3]:
DATASETS = {
    #"DIOR": {"hf_name": "HichTala/dior"},
    "COCO": {"hf_name": "HichTala/coco"},
    "DOTA": {"hf_name": "HichTala/dota"},
    "DEEPFRUITS": {"hf_name": "HichTala/deepfruits"},
    "XVIEW": {"hf_name": "HichTala/xview"},
    "OKTOBERFEST": {"hf_name": "HichTala/oktoberfest"},
    "FASHIONPEDIA": {"hf_name": "HichTala/fashionpedia"},
    "CADOT": {"hf_name": "HichTala/cadot"},
    "ARTAXOR": {"hf_name": "HichTala/artaxor"},
    "UODD": {"hf_name": "HichTala/uodd"},
}

In [4]:

class HFStreamingImageDataset(IterableDataset):
    def __init__(self, dataset_name, split="train", image_col="image", transform=None):
        self.dataset_name = dataset_name
        self.split = split
        self.image_col = image_col
        self.transform = transform

    def __iter__(self):
        stream_ds = load_dataset(
            self.dataset_name,
            split=self.split,
            streaming=True
        )

        for item in stream_ds:
            img = item[self.image_col]

            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)

            img = img.convert("RGB")

            if self.transform:
                img = self.transform(img)

            yield img



In [5]:
dataset = HFStreamingImageDataset(
    dataset_name="your_dataset_name",
    split="train",
    image_col="image",
    transform=preprocess
)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    num_workers=2,  
    pin_memory=True
)


In [6]:
@torch.no_grad()
def extract_embeddings_streaming(
    model,
    dataloader,
    device,
    output_dir,
    prefix="emb"
):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    idx = 0
    for images in tqdm(dataloader, desc="Extracting embeddings"):
        images = images.to(device, non_blocking=True)
        feats = model(images)  

        feats = feats.cpu().numpy().astype(np.float16)

        for emb in feats:
            np.save(
                os.path.join(output_dir, f"{prefix}_{idx}.npy"),
                emb
            )
            idx += 1



In [7]:
def build_embeddings_for_datasets_streaming(
    configs: dict,
    model,
    preprocess,
    device,
    batch_size: int = 64,
    num_workers: int = 0,
    save_dir: str = "results",
):
    model.eval()

    # Dossier "results" à la racine du projet (là où est le notebook)
    project_root = os.getcwd()
    save_dir = os.path.join(project_root, save_dir)
    os.makedirs(save_dir, exist_ok=True)

    for key, cfg in configs.items():
        if not cfg.get("enabled", True):
            continue

        print(f"\n▶ Processing dataset: {key}")

        dataset = HFStreamingImageDataset(
            dataset_name=cfg["hf_name"],
            split=cfg.get("split", "train"),
            image_col=cfg.get("image_col", "image"),
            transform=preprocess
        )

        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=True
        )

        out_dir = os.path.join(save_dir, key)
        os.makedirs(out_dir, exist_ok=True)

        idx = 0
        with torch.no_grad():
            for images in tqdm(dataloader, desc=f"Embedding {key}"):
                images = images.to(device, non_blocking=True)
                feats = model(images)
                feats = feats.cpu().numpy().astype(np.float16)

                for emb in feats:
                    np.save(
                        os.path.join(out_dir, f"{key}_{idx:08d}.npy"),
                        emb
                    )
                    idx += 1

        print(f"✔ {key}: {idx} embeddings saved in {out_dir}")


In [8]:
build_embeddings_for_datasets_streaming(
    configs=DATASETS,
    model=model,
    preprocess=preprocess,
    device=device,
    batch_size=64,
    num_workers=0,
    save_dir="results"
)



▶ Processing dataset: COCO


Embedding COCO: 0it [00:00, ?it/s]/opt/miniconda3/envs/image_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Embedding COCO: 1833it [1:21:08,  2.66s/it]


✔ COCO: 117266 embeddings saved in /Users/cyprienvial/Documents/3A/Image/Etude_technique/results/COCO

▶ Processing dataset: DOTA


Embedding DOTA: 0it [00:00, ?it/s]/opt/miniconda3/envs/image_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Embedding DOTA: 846it [09:40,  1.46it/s]


✔ DOTA: 54087 embeddings saved in /Users/cyprienvial/Documents/3A/Image/Etude_technique/results/DOTA

▶ Processing dataset: DEEPFRUITS


...

✔ CADOT: 3234 embeddings saved in /Users/cyprienvial/Documents/3A/Image/Etude_technique/results/CADOT

▶ Processing dataset: ARTAXOR


Embedding ARTAXOR: 193it [09:44,  3.03s/it]


✔ ARTAXOR: 12300 embeddings saved in /Users/cyprienvial/Documents/3A/Image/Etude_technique/results/ARTAXOR

▶ Processing dataset: UODD


Embedding UODD: 40it [00:28,  1.40it/s]

✔ UODD: 2560 embeddings saved in /Users/cyprienvial/Documents/3A/Image/Etude_technique/results/UODD
